> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


In [1]:
from mrta.evaluation.eval_pipeline import run_eval
from mrta.evaluation.metrics import (
    answer_relevance,
    citation_correctness,
    faithfulness,
    hallucination_rate,
)
from mrta.observability.logging import StructuredLogger

# Phase 9 — Evaluation, Observability & Docker

**Goal:** Make the project look senior-level. We add three production-shaped layers on top of the working RAG system:

1. **Evaluation** — a small benchmark + Ragas metrics for groundedness, faithfulness, and answer relevance.
2. **Observability** — structured JSONL logs with per-run latency, tokens, cited pages, and retrieval traces.
3. **Deployment** — a `docker-compose.yml` that brings up backend + frontend + Qdrant with one command.

Most candidates skip these. That is exactly why doing them is portfolio gold.


## 9.1 — Build a tiny benchmark

Five hand-labeled QA pairs over *Attention Is All You Need*. In a real project you would have 50-200; this is enough to show the pipeline.


In [3]:
import os, sys, json
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

BENCH = Path("data/processed/benchmark.jsonl")
BENCH.parent.mkdir(parents=True, exist_ok=True)

benchmark = [
    {
        "question": "What is the dimensionality of the model and the feed-forward inner layer in the base Transformer?",
        "expected_pages": [3, 5],
        "expected_substrings": ["512", "2048"],
    },
    {
        "question": "How many attention heads does the base Transformer use?",
        "expected_pages": [5],
        "expected_substrings": ["8"],
    },
    {
        "question": "What positional encoding does the Transformer use and why?",
        "expected_pages": [6],
        "expected_substrings": ["sinusoid"],
    },
    {
        "question": "What BLEU score did the base Transformer achieve on WMT 2014 English-to-German?",
        "expected_pages": [8],
        "expected_substrings": ["27.3", "BLEU"],
    },
    {
        "question": "Why is scaled dot-product attention scaled by 1/sqrt(d_k)?",
        "expected_pages": [4],
        "expected_substrings": ["soft", "magnitude"],
    },
]

with BENCH.open("w") as f:
    for r in benchmark:
        f.write(json.dumps(r) + "\n")
print(f"Wrote {len(benchmark)} questions to {BENCH}")


Wrote 5 questions to data/processed/benchmark.jsonl


## 9.2 — Custom metrics (no external deps)

Start with metrics you can compute yourself. They are interpretable and need no LLM judge.


In [4]:
from mrta.core.llm import LLMClient
from mrta.core.rag_pipeline import rag_query
from mrta.evaluation.metrics import (
    answer_relevance,
    citation_correctness,
    faithfulness,
    hallucination_rate,
)
from mrta.retrieval.embedder import Embedder
from mrta.retrieval.vector_store import VectorStore

embedder = Embedder()
store = VectorStore.load("data/vector_store/aiayn", embedder)  # requires pre-built index
llm = LLMClient()
print("Components loaded.")

Components loaded.


## 9.3 — Run the benchmark


In [ ]:
import pandas as pd

rows = []
for q in benchmark:
    result = rag_query(q["question"], store, llm)
    rows.append({
        "question": q["question"][:60] + "...",
        "answer_relevance": round(answer_relevance(q["question"], result["answer"]), 2),
        "faithfulness": round(faithfulness(result["answer"], result["sources"]), 2),
        "citation_correctness": round(citation_correctness(result["answer"], result["sources"]), 2),
        "hallucination_rate": round(hallucination_rate(result["answer"], result["sources"]), 2),
        "latency_s": round(result["latency_s"], 1),
    })
df = pd.DataFrame(rows)
df

,question,answer_relevance,faithfulness,citation_correctness,hallucination_rate,latency_s
0,What is the dimensionality of the model and th...,0.71,0.5,1.0,0.5,2.3
1,How many attention heads does the base Transfo...,0.83,1.0,1.0,0.0,1.8
2,What positional encoding does the Transformer ...,1.00,1.0,1.0,0.0,2.2
3,What BLEU score did the base Transformer achie...,1.00,1.0,0.0,0.0,1.8
4,Why is scaled dot-product attention scaled by ...,0.75,1.0,1.0,0.0,2.9


: 

## 9.4 — Ragas for LLM-judged groundedness

[Ragas](https://github.com/explodinggradients/ragas) computes:

- **faithfulness** — every claim in the answer is supported by the context.
- **answer_relevancy** — the answer addresses the question.
- **context_precision** — retrieved chunks are useful.

It uses an LLM as judge. We point it at Ollama to keep everything local.

> Note: Ragas wraps several LangChain components. The exact setup snippet below is the simplest illustrative path; the version of Ragas you install may have a slightly different API surface, so consult its docs.


In [ ]:
# Illustrative — uncomment and adapt to your installed ragas version.
# from datasets import Dataset
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision
# from langchain_community.chat_models import ChatOllama
# from langchain_community.embeddings import HuggingFaceEmbeddings
#
# judge_llm = ChatOllama(model="llama3.2:3b", temperature=0)
# judge_emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
#
# rows = []
# for q in benchmark:
#     res = run_question(q["question"])
#     ctx = [store.metadata[i]["text"] for i in range(min(5, len(store.metadata)))]
#     rows.append({"question": q["question"], "answer": res["answer"], "contexts": ctx, "ground_truth": ""})
#
# ds = Dataset.from_list(rows)
# scores = evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision],
#                   llm=judge_llm, embeddings=judge_emb)
# print(scores)
print("See the cell above for the Ragas template; uncomment and adapt for your installed version.")


## 9.5 — Structured logging

Every production run goes through one logger. The format is JSONL: one event per line, machine-parseable, append-only.


In [ ]:
from mrta.observability.logging import StructuredLogger

logger = StructuredLogger()
# Usage after rag_query:
# result = rag_query(question, store, llm)
# logger.log_run(
#     question=question,
#     answer=result["answer"],
#     sources=result["sources"],
#     latency_s=result["latency_s"],
# )
print("StructuredLogger ready — logs append to", __import__("mrta").settings.log_file)

StructuredLogger ready — logs append to data/logs/runs.jsonl


## 9.6 — Aggregating logs

A 5-line analyzer that turns the JSONL into a dashboard-ready DataFrame:


In [ ]:
import pandas as pd

events = [json.loads(line) for line in LOG_FILE.read_text().splitlines()]
df_logs = pd.DataFrame(events)
print("Total runs logged:", len(df_logs))
df_logs[["timestamp", "question", "latency_s", "cited_pages"]].tail(5)


## 9.7 — Dockerfiles

Two Dockerfiles — one per process. Keeping them separate lets you scale the LLM-heavy API independently of the UI.


In [ ]:
# Dockerfiles already exist — see docker/Dockerfile.api and docker/Dockerfile.ui
# Built and validated in Phase 6.

## 9.8 — docker-compose for the full stack

Three services:

- **api** — FastAPI app.
- **ui** — Streamlit frontend.
- **qdrant** — vector store for the production swap.

Ollama runs on the host (not in a container) because GPU pass-through is platform-specific and `ollama serve` on the host already speaks HTTP.


In [ ]:
# docker-compose already exists — see docker/docker-compose.yml
# Brings up api + ui + qdrant with: docker compose -f docker/docker-compose.yml up

## 9.9 — The Qdrant swap

Once the vector store is Qdrant, the `VectorStore` class only changes inside its methods — the public API stays the same. That is the payoff for designing an interface in Phase 3 rather than calling FAISS directly from the RAG pipeline.

Sketch:

```python
from qdrant_client import QdrantClient, models

class QdrantStore:
    def __init__(self, url: str, dim: int, embedder, collection: str = "papers"):
        self.client = QdrantClient(url=url)
        if collection not in [c.name for c in self.client.get_collections().collections]:
            self.client.create_collection(
                collection_name=collection,
                vectors_config=models.VectorParams(size=dim, distance=models.Distance.COSINE),
            )
        self.collection = collection
        self.embedder = embedder

    def add(self, chunks):
        embs = self.embedder.encode([c.text for c in chunks], normalize_embeddings=True)
        self.client.upsert(self.collection, points=[
            models.PointStruct(id=i, vector=v.tolist(), payload=c.model_dump())
            for i, (c, v) in enumerate(zip(chunks, embs))
        ])

    def search(self, query: str, k: int = 5, doc_id: str | None = None):
        q = self.embedder.encode([query], normalize_embeddings=True)[0].tolist()
        flt = models.Filter(must=[models.FieldCondition(key="doc_id", match=models.MatchValue(value=doc_id))]) if doc_id else None
        res = self.client.search(self.collection, query_vector=q, query_filter=flt, limit=k)
        return [{**r.payload, "score": r.score} for r in res]
```

Now the `VECTOR_STORE` env var swaps backends with no callsite changes.


## 9.10 — CI on GitHub Actions

Drop this file at `.github/workflows/ci.yml`:


In [ ]:
# CI workflow already exists — see .github/workflows/ci.yml
# Runs on every push/PR to main: ruff → black --check → pytest

## 9.11 — README polish checklist

A senior-quality README hits all of these. Tick them off before announcing the repo:

- [ ] One-sentence elevator pitch at the top.
- [ ] A screenshot or animated GIF of the app working.
- [ ] An architecture diagram (Excalidraw or draw.io). Save as `docs/architecture.png`.
- [ ] A "Why these choices?" section — FAISS vs Qdrant, local vs API, chunk size.
- [ ] A reproducible **Quick start** that gets a stranger to a running demo in <5 minutes.
- [ ] An evaluation section with at least one table of metrics.
- [ ] A "Limitations" section that names what you would do next.
- [ ] A demo video link (5–8 min walkthrough on YouTube/Loom).

Honesty about limitations is the strongest senior signal — it shows you know where the project would break.


## What you learned

- A custom-metrics evaluation harness with `recall@k`, substring grounding, and citation-in-retrieval rate.
- How Ragas plugs an LLM judge into the pipeline (and that you should use a local Ollama judge to stay free).
- Structured JSONL logging with `structlog` for replayable runs.
- A two-Dockerfile + `docker-compose.yml` setup with Qdrant as the production vector store.
- The Qdrant swap sketch — payoff for designing a clean `VectorStore` interface earlier.
- A GitHub Actions CI workflow.
- A README polish checklist.

## Exercises

1. Expand the benchmark to 30 questions across 3 papers (e.g. AIAYN, BERT, GPT-2 / GPT-3).
2. Add a `latency-p95` SLO to CI: fail the build if it regresses above 10s.
3. Pipe the JSONL logs into a small Streamlit dashboard (`docs/dashboard.py`) that shows recall, latency, and cost over time.
4. Record a 5-minute demo video and link it from the README.

---

That's the full tutorial series. You now have:

1. A repo scaffold with `src/mrta/`, `apps/`, `docker/`, CI, and typed config.
2. Ten runnable tutorial notebooks (`notebooks/2026-05-25-phase00 ... phase09`).
3. An end-to-end working multimodal RAG assistant on local models.
4. The eval, observability, and deployment scaffolding to justify the "production-ready" label on your README.

Good luck — go ship it.
